# **1. Imports**

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from torchvision import transforms as T
from sklearn.utils.class_weight import compute_class_weight
import wandb
import optuna
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt 

# **2. Convolutional Neural Network Structure**

![alt text](CnnImage.webp) 

### **2.1 CNN Simple**

CNN with **3 convolutional layers** and **2 fully-connected layers**

In [2]:
class CNN_Simple(nn.Module):
    def __init__(self, in_channels=1, num_classes=3, 
                 normalize2d=nn.BatchNorm2d, normalize1d=nn.BatchNorm1d, 
                 pool=nn.MaxPool2d, 
                 drop_conv_prob=0.2, drop_fc_prob=0.4, 
                 activation=nn.ReLU()):
        super(CNN_Simple, self).__init__()

        self.act = activation
        self.pool = pool(kernel_size=2, stride=2)
        self.drop2d = nn.Dropout2d(drop_conv_prob)

        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = normalize2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = normalize2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn3 = normalize2d(128)

        self.globavgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(128, 128)
        self.bn_fc1 = normalize1d(128)
        self.drop = nn.Dropout(drop_fc_prob)

        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(self.drop2d(self.act(self.bn1(self.conv1(x)))))
        x = self.pool(self.drop2d(self.act(self.bn2(self.conv2(x)))))
        x = self.pool(self.drop2d(self.act(self.bn3(self.conv3(x)))))

        x = self.globavgpool(x)
        x = x.view(x.size(0), -1)
        
        x = self.drop(self.act(self.bn_fc1(self.fc1(x))))
        x = self.fc2(x)
        return x

### **2.2 CNN One Fully Connected**

CNN with **3 convolutional layers** and **1 fully-connected layer**. The hyperparameters associated with normalisation and dropout regarding the fully-connected are not used, but were kept to preserve a common interface between models.

In [3]:
class CNN_1FC(nn.Module):
    def __init__(self, in_channels=1, num_classes=3, 
                 normalize2d=nn.BatchNorm2d, normalize1d=nn.BatchNorm1d, 
                 pool=nn.MaxPool2d, 
                 drop_conv_prob=0.2, drop_fc_prob=0.4, 
                 activation=nn.ReLU()):
        super(CNN_1FC, self).__init__()

        self.act = activation
        self.pool = pool(kernel_size=2, stride=2)
        self.drop2d = nn.Dropout2d(drop_conv_prob)

        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = normalize2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = normalize2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn3 = normalize2d(128)

        self.globavgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(self.drop2d(self.act(self.bn1(self.conv1(x)))))
        x = self.pool(self.drop2d(self.act(self.bn2(self.conv2(x)))))
        x = self.pool(self.drop2d(self.act(self.bn3(self.conv3(x)))))

        x = self.globavgpool(x)
        x = x.view(x.size(0), -1)
        
        x = self.fc1(x)
        return x

### **2.3 CNN VGG-Like**

In [4]:
class CNN_VGG(nn.Module):
    def __init__(self, in_channels=1, num_classes=3,
                 normalize2d=nn.BatchNorm2d, normalize1d=nn.BatchNorm1d,
                 pool=nn.MaxPool2d,
                 drop_conv_prob=0.2, drop_fc_prob=0.4,
                 activation=nn.ReLU()):
        
        super(CNN_VGG, self).__init__()

        self.act = activation
        self.pool = pool(kernel_size=2, stride=2)
        self.drop2d = nn.Dropout2d(drop_conv_prob)
        
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = normalize2d(32)
        self.conv1_2 = nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1)
        self.bn1_2 = normalize2d(32)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = normalize2d(64)
        self.conv2_2 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)
        self.bn2_2 = normalize2d(64)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn3 = normalize2d(128)
        self.conv3_2 = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
        self.bn3_2 = normalize2d(128)

        self.globavgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.fc1 = nn.Linear(128, 128)
        self.bn_fc1 = normalize1d(128)

        self.drop = nn.Dropout(drop_fc_prob)

        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(self.drop2d(self.act(self.bn1(self.conv1(x)))))
        x = self.pool(self.drop2d(self.act(self.bn1_2(self.conv1_2(x)))))

        x = self.pool(self.drop2d(self.act(self.bn2(self.conv2(x)))))
        x = self.pool(self.drop2d(self.act(self.bn2_2(self.conv2_2(x)))))

        x = self.pool(self.drop2d(self.act(self.bn3(self.conv3(x)))))
        x = self.pool(self.drop2d(self.act(self.bn3_2(self.conv3_2(x)))))

        x = self.globavgpool(x)
        x = x.view(x.size(0), -1)

        x = self.drop(self.act(self.bn_fc1(self.fc1(x))))
        x = self.fc2(x)
        return x

### **2.4 CNN Hybrid**

In [5]:
class CNN_Hybrid(nn.Module):
    def __init__(self, in_channels=1, num_classes=3,
                 normalize2d=nn.BatchNorm2d, normalize1d=nn.BatchNorm1d,
                 pool=nn.MaxPool2d,
                 drop_conv_prob=0.2, drop_fc_prob=0.4,
                 activation=nn.ReLU()):
        super(CNN_Hybrid, self).__init__()

        self.act = activation
        self.pool = pool(kernel_size=2, stride=2)
        self.drop2d = nn.Dropout2d(drop_conv_prob)

        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = normalize2d(32)
        self.conv1_2 = nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1)
        self.bn1_2 = normalize2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = normalize2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn3 = normalize2d(128)

        self.globavgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(128, 128)
        self.bn_fc1 = normalize1d(128)
        
        self.drop = nn.Dropout(drop_fc_prob)

        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(self.drop2d(self.act(self.bn1(self.conv1(x)))))
        x = self.pool(self.drop2d(self.act(self.bn1_2(self.conv1_2(x)))))

        x = self.pool(self.drop2d(self.act(self.bn2(self.conv2(x)))))

        x = self.pool(self.drop2d(self.act(self.bn3(self.conv3(x)))))

        x = self.globavgpool(x)
        x = x.view(x.size(0), -1)

        x = self.drop(self.act(self.bn_fc1(self.fc1(x))))
        x = self.fc2(x)
        return x

### **2.5 CNN Deep**

In [6]:
class CNN_Deep(nn.Module):
    def __init__(self, in_channels=1, num_classes=3,
                 normalize2d=nn.BatchNorm2d, normalize1d=nn.BatchNorm1d,
                 pool=nn.MaxPool2d,
                 drop_conv_prob=0.2, drop_fc_prob=0.4,
                 activation=nn.ReLU()):
        super(CNN_Deep, self).__init__()

        self.act = activation
        self.pool = pool(kernel_size=2, stride=2)
        self.drop2d = nn.Dropout2d(drop_conv_prob)

        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = normalize2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = normalize2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn3 = normalize2d(128)

        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.bn4 = normalize2d(256)

        self.globavgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(256, 128)
        self.bn_fc1 = normalize1d(128)

        self.drop = nn.Dropout(drop_fc_prob)
        
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(self.act(self.bn1(self.conv1(x))))
        x = self.pool(self.act(self.bn2(self.conv2(x))))
        x = self.pool(self.act(self.bn3(self.conv3(x))))
        x = self.pool(self.act(self.bn4(self.conv4(x))))
    
        x = self.globavgpool(x)
        x = x.view(x.size(0), -1)

        x = self.drop(self.act(self.bn_fc1(self.fc1(x))))
        x = self.fc2(x)
        return x


# **3. Dataset Preparation**

### **3.1 Class for images**

In [7]:
class ImageDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        image = Image.open(img_path).convert('L')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)
            
        return image, label

### **3.2 Data augmentation**

In [8]:
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(30),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

### **3.3 Report of the dataset**

In [9]:
data_dir = Path('Breast-Cancer-Dataset')
classes = [d.name for d in data_dir.iterdir() if d.is_dir()]
label_map = {name: i for i, name in enumerate(classes)}
num_classes = len(classes)

print(f"Found classes: {label_map}")

file_paths = []
labels = []
for class_name, label_idx in label_map.items():
    class_dir = data_dir / class_name
    for img_path in class_dir.glob('*.[jp][pn]g'): 
        file_paths.append(str(img_path))
        labels.append(label_idx)

print(f"Total images found: {len(file_paths)}")

Found classes: {'benign': 0, 'malignant': 1, 'normal': 2}
Total images found: 780


### **3.4 Train-Test Split**

In [10]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, 
    labels, 
    test_size=0.2,
    random_state=42,
    stratify=labels
)

train_dataset = ImageDataset(train_paths, train_labels, transform=train_transform)
val_dataset = ImageDataset(val_paths, val_labels, transform=val_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")

Train images: 624
Validation images: 156


# **4. Training and Validation**

### **4.1 Wandb Login**

We will use [**Weights and Biases'**](https://wandb.ai/site/) visualization to analyze various metrics and also load different models. It is **necessary** to add your own *WANDB_API_KEY*.

In [ ]:
wandb.login(key="")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Usuario\_netrc
wandb: Currently logged in as: juand24601 (juand24601-university-of-las-palmas-de-gran-canaria) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### **4.2 Device**

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(42)
np.random.seed(42)

Using device: cuda


### **4.3 Balance Classes' Weights**

Since our BUSI Dataset is imbalanced we will fix this by adding more weight to the minority class

In [13]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

### **4.4 Options for optimizers, activations, and hyperparameters to study**

If you wish to add other options, bear in mind that some optimizers, activations, or whatever else you choose **work differently** (for example, *SGD needs momentum*, so we have added the necessary code to work with the SGD optimizer).

In [14]:
ARCHITECTURES = {
    "CNN_Simple": CNN_Simple,
    "CNN_1FC": CNN_1FC,
    "CNN_VGG": CNN_VGG,
    "CNN_Hybrid": CNN_Hybrid,
    "CNN_Deep": CNN_Deep,
}

OPTIMIZERS = {
    "AdamW": torch.optim.AdamW,
    "Adam": torch.optim.Adam,
    "NAdam": torch.optim.NAdam,
    "SGD": torch.optim.SGD,
}

ACTIVACTIONS = {
    "ReLU": nn.ReLU(),
    "LeakyReLU": nn.LeakyReLU(),
}

NORMALIZATIONS_2D = {
    "BatchNorm2d": nn.BatchNorm2d,
    "InstanceNorm2d": nn.InstanceNorm2d,
}

NORMALIZATIONS_1D = {
    "BatchNorm1d": nn.BatchNorm1d,
    "InstanceNorm1d": nn.InstanceNorm1d,}

POOLS = {
    "MaxPool2d": nn.MaxPool2d,
    "AvgPool2d": nn.AvgPool2d,
}


### **4.5 Training and Validation loop (with Optuna)**

We will use [**Optuna**](https://optuna.org/), as it is a hyperparameter optimization framework that **automatically searches for the best combination of hyperparameters**, instead of manually testing different learning rates, dropout rates, optimizers, etc.

Optuna intelligently explores hyperparameter options, each test trains a model with different hyperparameters, and Optuna learns from previous tests to suggest better configurations, in our case **efficiently maximizing our validation F1 score**.

All Optuna tests will be visualized in Wandb.

In [15]:
# ====================== OPTUNA'S OBJECTIVE ======================
def objective(trial):

    # ------------------- HYPERPARAMETERS TO OPTIMIZE -------------------
    lr = trial.suggest_float("learning_rate", 0.0001, 0.005, log=True)
    drop_fc = trial.suggest_float("dropout", 0.2, 0.8)
    drop_conv = trial.suggest_float("dropout_conv", 0.0, 0.4)
    weight_decay = trial.suggest_float("weight_decay", 0.000001, 0.001, log=True)
    patience = trial.suggest_int("patience", 5, 12)
    opti = trial.suggest_categorical("optimizer", list(OPTIMIZERS.keys()))
    activ_func = trial.suggest_categorical("activ_func", list(ACTIVACTIONS.keys()))
    normalize_2d = trial.suggest_categorical("normalize_2d", list(NORMALIZATIONS_2D.keys()))
    normalize_1d = trial.suggest_categorical("normalize_1d", list(NORMALIZATIONS_1D.keys()))
    pools = trial.suggest_categorical("pool", list(POOLS.keys()))
    architecture = trial.suggest_categorical("architecture", list(ARCHITECTURES.keys()))

    # ------------------- INITIALIZE WANDB FOR TRIAL -------------------
    wandb.init(
        project="BUSI-CNN-OPTUNA-BEST-F1",
        name=f"trial-{trial.number}",
        config={
            "learning_rate": lr,
            "weight_decay": weight_decay,
            "patience": patience,
            "epochs": 50,
            "optimizer": opti,
            "activation_function": activ_func,
            "normalize_2d": normalize_2d,
            "normalize_1d": normalize_1d,
            "pool": pools,
            "dropout": drop_fc,
            "dropout_conv": drop_conv,
            "architecture": architecture
        }
    )
    config = wandb.config

    # ------------------- MODEL -------------------
    model = ARCHITECTURES[architecture](
        in_channels=1,
        num_classes=3,
        normalize2d=NORMALIZATIONS_2D[normalize_2d],
        normalize1d=NORMALIZATIONS_1D[normalize_1d],
        pool=POOLS[pools],
        activation=ACTIVACTIONS[activ_func],
        drop_conv_prob=drop_conv,
        drop_fc_prob=drop_fc
        ).to(device)

    # ------------------- OPTIMIZER -------------------

    if opti == "SGD":
        optimizer = OPTIMIZERS[opti](
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
            momentum=0.9
        )

    else:
        optimizer = OPTIMIZERS[opti](
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # ------------------- LOSS FUNCTION WITH CLASS WEIGHTS -------------------
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

    # ------------------- LEARNING RATE SCHEDULER -------------------
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.1,
        patience=patience,
    )

    # ------------------- TRAINING -------------------
    best_val_loss = float("inf")
    patience_counter = 0
    NUM_EPOCHS = config.epochs

    for epoch in range(NUM_EPOCHS):

        # -------- TRAIN --------
        model.train()
        train_loss_sum, correct_train, n_train = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * images.size(0)
            _, pred = outputs.max(1)
            n_train += labels.size(0)
            correct_train += (pred == labels).sum().item()

        train_loss = train_loss_sum / n_train
        train_acc = 100 * correct_train / n_train

        # -------- VAL --------
        model.eval()
        val_loss_sum, correct_val, n_val = 0, 0, 0

        all_preds, all_labels = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss_sum += loss.item() * images.size(0)
                _, pred = outputs.max(1)
                n_val += labels.size(0)
                correct_val += (pred == labels).sum().item()

                all_preds.extend(pred.cpu().numpy())
                all_labels.extend(labels.cpu().numpy()) 

        val_loss = val_loss_sum / n_val
        val_acc = 100 * correct_val / n_val

        f1 = f1_score(all_labels, all_preds, average='weighted')

        scheduler.step(val_loss)

        trial.report(val_loss, epoch)

        wandb.log({
            "epoch": epoch,
            "train/loss": train_loss,
            "train/acc": train_acc,
            "val/loss": val_loss,
            "val/acc": val_acc,
            "val/f1_score": f1,
            "optimizer": opti,
            "lr": optimizer.param_groups[0]['lr']
        })

        # -------- EARLY STOPPING --------
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # =================== CONFUSION MATRIX  ===================
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    wandb.log({
        "confusion_matrix": wandb.plot.confusion_matrix(
            y_true=all_labels,
            preds=all_preds,
            class_names=["benign", "malignant", "normal"]
        )
    })
    
    wandb.finish()
    return f1

# ====================== OPTUNA'S STUDY ======================
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Best trial:", study.best_trial.params)

[I 2025-12-19 20:37:02,058] A new study created in memory with name: no-name-60b2ded7-a65f-41ea-9ed8-a7358fda694f


epoch,▁▂▂▃▃▄▄▅▅▆▆▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▃▃▃▅▄▇▁▆█▄▄▄▄▆
train/loss,██▅▅▃▃▅▁▃▁▄▂▂▁
val/acc,▅▅▆▂▂▄▂▁▄▂▅▇█▆
val/f1_score,▁▄▅▂▃▃▃▅▅▃▅▆█▅
val/loss,█▇▁▅▇▄▂▄▄▄▅▅▄▆
epoch,13
lr,0.00231
optimizer,SGD
train/acc,37.01923


[I 2025-12-19 20:38:48,443] Trial 0 finished with value: 0.4839824922296131 and parameters: {'learning_rate': 0.002312709018987351, 'dropout': 0.49892453147733334, 'dropout_conv': 0.3431493763601914, 'weight_decay': 4.646447744591687e-05, 'patience': 11, 'optimizer': 'SGD', 'activ_func': 'LeakyReLU', 'normalize_2d': 'InstanceNorm2d', 'normalize_1d': 'BatchNorm1d', 'pool': 'MaxPool2d', 'architecture': 'CNN_VGG'}. Best is trial 0 with value: 0.4839824922296131.


epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▁▂▅▄▄▄▅▅▅▅▆▆▅▄▂▅▅▆▇▅▆▇▆▆▅▇▇▆▇▆▆▅▆▆▆▆█▆▆
train/loss,███▅▆▅▅▅▄▅▃▃▄▃▄▃▃▂▃▄▃▂▂▂▂▂▃▂▂▂▂▂▂▂▂▂▂▁▂▁
val/acc,▁▄▅▇▇█▆▇█▆▅▇▄▅▇██▆▅▆▆█▇▅▄▆▅▅▆▆▇▆█▆▅▅▆▅▅▅
val/f1_score,▁▂▆▇█▆▇█▆▁▃▇▃▃▆▇█▅▃▅▅█▆▃▂▅▃▃▅▅▆▆█▄▄▄▅▄▃▃
val/loss,█▇▆▅▄▅▄▄▄▄▄▃▃▄▃▂▂▃▂▂▂▃▃▂▂▂▂▃▂▁▂▁▁▁▂▂▁▂▂▁
epoch,49
lr,0.00057
optimizer,AdamW
train/acc,47.4359


[I 2025-12-19 20:44:38,758] Trial 1 finished with value: 0.3029737806053595 and parameters: {'learning_rate': 0.0005746320658842209, 'dropout': 0.4598313909620014, 'dropout_conv': 0.1398262314828604, 'weight_decay': 1.6468294488675764e-05, 'patience': 10, 'optimizer': 'AdamW', 'activ_func': 'ReLU', 'normalize_2d': 'BatchNorm2d', 'normalize_1d': 'BatchNorm1d', 'pool': 'AvgPool2d', 'architecture': 'CNN_Simple'}. Best is trial 0 with value: 0.4839824922296131.


...

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▂▂▁▃▁▃▃▃▄▄▄▄▅▄▅▅▄▄▅▅▅▆▅▅▅▇▆▆▆▇▇▇▇▆▇▇██▇█
train/loss,█▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▅▄▅▄▄▄▄▅▃▄▃▄▃▂▃▂▃▂▃▂▁▁▂▁
val/acc,▁▃▆▂▆▅▃▅▅▆▆▅▅▆▄▅▆▆▆▆▆▆▇▇▆▇▆▆█▇▆▇█▇▇█▇▇▇█
val/f1_score,▁▃▅▁▆▅▃▅▅▄▄▆▅▆▆▆▆▆▆▆▇▇▆▆▇▇▆▇▇▇▇█▇▇███▇██
val/loss,██████▇▇▆▆▆▅▆▅▅▄▅▅▅▄▄▄▅▅▄▄▃▃▃▃▃▃▂▃▂▂▂▂▁▃
epoch,49
lr,0.00102
optimizer,AdamW
train/acc,70.03205


[I 2025-12-19 22:46:39,532] Trial 28 finished with value: 0.7861950549450549 and parameters: {'learning_rate': 0.0010220113200458546, 'dropout': 0.6771555477588658, 'dropout_conv': 0.07910781353774331, 'weight_decay': 0.00024755290875160907, 'patience': 12, 'optimizer': 'AdamW', 'activ_func': 'ReLU', 'normalize_2d': 'InstanceNorm2d', 'normalize_1d': 'InstanceNorm1d', 'pool': 'AvgPool2d', 'architecture': 'CNN_VGG'}. Best is trial 8 with value: 0.8650860273606176.


epoch,▁▂▂▃▄▄▅▅▆▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▃▁▃▄▇█▆▄▆▆▆▃
train/loss,█▇▄▄▄▂▃▂▁▂▂▃
val/acc,█▃▄▄▅▃▂▂▅▁▃▃
val/f1_score,█▄▅▄▆▄▂▃▆▁▄▄
val/loss,▁█▆▂▄▆▆▄▅▇▂▄
epoch,11
lr,0.00176
optimizer,SGD
train/acc,33.49359


[I 2025-12-19 22:47:58,970] Trial 29 finished with value: 0.18350202429149798 and parameters: {'learning_rate': 0.0017604917372532386, 'dropout': 0.6215155694837169, 'dropout_conv': 0.31667454553507113, 'weight_decay': 4.714477142645431e-05, 'patience': 11, 'optimizer': 'SGD', 'activ_func': 'LeakyReLU', 'normalize_2d': 'InstanceNorm2d', 'normalize_1d': 'BatchNorm1d', 'pool': 'MaxPool2d', 'architecture': 'CNN_VGG'}. Best is trial 8 with value: 0.8650860273606176.


Best trial: {'learning_rate': 0.0010581221441639968, 'dropout': 0.6697240967045921, 'dropout_conv': 0.16717031904520102, 'weight_decay': 8.661393093253565e-05, 'patience': 10, 'optimizer': 'AdamW', 'activ_func': 'ReLU', 'normalize_2d': 'InstanceNorm2d', 'normalize_1d': 'InstanceNorm1d', 'pool': 'MaxPool2d', 'architecture': 'CNN_VGG'}
